In [1]:
#imports
import kagglehub
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import brier_score_loss
import matplotlib.pyplot as plt


# Download latest version
path = kagglehub.dataset_download("nishaanamin/march-madness-data")

print("Path to dataset files:", path)

Path to dataset files: /Users/frankmclaughlin/.cache/kagglehub/datasets/nishaanamin/march-madness-data/versions/145


In [2]:
for file in os.listdir(path):
    print(file)

NameError: name 'os' is not defined

In [ ]:


# Replace 'filename.csv' with whatever shows up from the cell above
df = pd.read_csv(path + '/AP Poll Data.csv')
print(df.shape)      # how many rows and columns
print(df.head())     # first 5 rows
print(df.columns)    # all column names

In [ ]:
df_2 = pd.read_csv(path + '/Tournament Matchups.csv')
print(df.head)

In [ ]:
print(df_2.columns.tolist())
print(df_2.head(3).to_string())

In [ ]:
df_bart = pd.read_csv(path + '/KenPom Barttorvik.csv')
print(df_bart.shape)
print(df_bart.columns.tolist())
print(df_bart.head(3).to_string())

df_538 = pd.read_csv(path + '/538 Ratings.csv')
print(df_538.shape)
print(df_538.columns.tolist())
print(df_538.head(3).to_string())

In [ ]:
import pandas as pd
import numpy as np

# ── 1. Load raw files ──────────────────────────────────────────────────────────
df_matchups = pd.read_csv(path + '/Tournament Matchups.csv')
df_bart     = pd.read_csv(path + '/KenPom Barttorvik.csv')
df_538      = pd.read_csv(path + '/538 Ratings.csv')

# ── 2. Pair teams into matchups ────────────────────────────────────────────────
df_matchups = df_matchups.sort_values(['YEAR', 'BY YEAR NO']).reset_index(drop=True)

team_a = df_matchups.iloc[0::2].reset_index(drop=True)
team_b = df_matchups.iloc[1::2].reset_index(drop=True)

games = pd.DataFrame({
    'YEAR'          : team_a['YEAR'],
    'GAME_ID'       : team_a['BY YEAR NO'],
    'CURRENT_ROUND' : team_a['CURRENT ROUND'],
    'TEAM_A'        : team_a['TEAM'],
    'TEAM_A_NO'     : team_a['TEAM NO'],
    'SEED_A'        : team_a['SEED'],
    'SCORE_A'       : team_a['SCORE'],
    'TEAM_B'        : team_b['TEAM'],
    'TEAM_B_NO'     : team_b['TEAM NO'],
    'SEED_B'        : team_b['SEED'],
    'SCORE_B'       : team_b['SCORE'],
})
games['TARGET'] = (games['SCORE_A'] > games['SCORE_B']).astype(int)
print(f"Games built: {len(games)}")

# ── 3. Barttorvik features (drop SEED to avoid collision) ─────────────────────
bart_cols = [
    'YEAR', 'TEAM NO',
    'KADJ EM', 'KADJ O', 'KADJ D',
    'BADJ EM', 'BADJ O', 'BADJ D',
    'BARTHAG', 'EFG%', 'EFG%D',
    'TOV%', 'TOV%D', 'OREB%', 'DREB%',
    'FTR', 'FTRD', 'ELITE SOS', 'WAB',
    'EXP', 'TALENT', 'WIN%',
]
df_bart_slim = df_bart[bart_cols].copy()

# ── 4. Merge Barttorvik onto team A and B ──────────────────────────────────────
games = games.merge(
    df_bart_slim.add_suffix('_A').rename(columns={'YEAR_A':'YEAR', 'TEAM NO_A':'TEAM_A_NO'}),
    on=['YEAR', 'TEAM_A_NO'], how='left'
)
games = games.merge(
    df_bart_slim.add_suffix('_B').rename(columns={'YEAR_B':'YEAR', 'TEAM NO_B':'TEAM_A_NO'}),
    left_on=['YEAR', 'TEAM_B_NO'], right_on=['YEAR', 'TEAM_A_NO'], how='left',
    suffixes=('', '_drop')
)
# Drop the duplicate key col from second merge
games = games.drop(columns=[c for c in games.columns if c.endswith('_drop') or c == 'TEAM_A_NO_y'], errors='ignore')

# ── 5. Merge 538 power rating ──────────────────────────────────────────────────
df_538_slim = df_538[['YEAR', 'TEAM NO', 'POWER RATING']].copy()

games = games.merge(
    df_538_slim.rename(columns={'TEAM NO':'TEAM_A_NO', 'POWER RATING':'POWER_RATING_A'}),
    on=['YEAR', 'TEAM_A_NO'], how='left'
)
games = games.merge(
    df_538_slim.rename(columns={'TEAM NO':'TEAM_B_NO', 'POWER RATING':'POWER_RATING_B'}),
    on=['YEAR', 'TEAM_B_NO'], how='left'
)

# ── 6. Difference features ─────────────────────────────────────────────────────
stat_bases = [
    'KADJ EM', 'KADJ O', 'KADJ D',
    'BADJ EM', 'BADJ O', 'BADJ D',
    'BARTHAG', 'EFG%', 'EFG%D',
    'TOV%', 'TOV%D', 'OREB%', 'DREB%',
    'ELITE SOS', 'WAB', 'WIN%', 'TALENT', 'EXP',
]
for stat in stat_bases:
    a, b = f'{stat}_A', f'{stat}_B'
    if a in games.columns and b in games.columns:
        games[f'{stat}_DIFF'] = games[a] - games[b]

games['SEED_DIFF']         = games['SEED_A'] - games['SEED_B']
games['POWER_RATING_DIFF'] = games['POWER_RATING_A'] - games['POWER_RATING_B']

# ── 7. Quality check ───────────────────────────────────────────────────────────
print(f"Final shape: {games.shape}")
print(f"Years: {sorted(games['YEAR'].unique())}")
print(f"Target distribution:\n{games['TARGET'].value_counts()}")
print(f"\nNull counts (top 10):\n{games.isnull().sum().sort_values(ascending=False).head(10)}")
print(games[['YEAR','TEAM_A','SEED_A','TEAM_B','SEED_B','TARGET',
             'KADJ EM_A','KADJ EM_B','KADJ EM_DIFF']].head(5).to_string())

In [ ]:
# ── 1. Drop 538 (too many nulls) and define feature set ───────────────────────
drop_cols = ['POWER_RATING_A', 'POWER_RATING_B', 'POWER_RATING_DIFF']
games_clean = games.drop(columns=drop_cols)

# Use only DIFF features + seeds for the model
# (raw A/B stats are redundant once we have diffs, and cause data leakage risk)
diff_features = [c for c in games_clean.columns if c.endswith('_DIFF')]
seed_features = ['SEED_A', 'SEED_B', 'SEED_DIFF']
feature_cols  = diff_features + seed_features

X = games_clean[feature_cols]
y = games_clean['TARGET']

print(f"Features: {len(feature_cols)}")
print(feature_cols)

# ── 2. Leave-one-year-out cross validation ────────────────────────────────────
# This is the RIGHT way to validate — never train on the same year you predict
years = sorted(games_clean['YEAR'].unique())
brier_scores = []
accuracies   = []

for test_year in years:
    train_mask = games_clean['YEAR'] != test_year
    test_mask  = games_clean['YEAR'] == test_year

    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)

    probs  = rf.predict_proba(X_test)[:, 1]
    preds  = rf.predict(X_test)
    brier  = brier_score_loss(y_test, probs)
    acc    = (preds == y_test).mean()

    brier_scores.append(brier)
    accuracies.append(acc)
    print(f"{test_year}  Brier: {brier:.4f}  Accuracy: {acc:.3f}  (n={test_mask.sum()})")

print(f"\nMean Brier: {np.mean(brier_scores):.4f}  (target < 0.20)")
print(f"Mean Accuracy: {np.mean(accuracies):.3f}")

# ── 3. Feature importance (train on all data except 2025) ─────────────────────
train_mask = games_clean['YEAR'] != 2025
rf_full = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf_full.fit(X[train_mask], y[train_mask])

importances = pd.Series(rf_full.feature_importances_, index=feature_cols) \
                .sort_values(ascending=False)

print("\nTop 15 features:")
print(importances.head(15))

# ── 4. Plot feature importances ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
importances.head(15).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title('Random Forest — Top 15 Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
top_features = [
    'BADJ EM_DIFF', 'KADJ EM_DIFF', 'BARTHAG_DIFF', 'WAB_DIFF',
    'TALENT_DIFF', 'KADJ O_DIFF', 'KADJ D_DIFF', 'EXP_DIFF',
    'BADJ O_DIFF', 'BADJ D_DIFF', 'TOV%D_DIFF', 'EFG%D_DIFF',
    'DREB%_DIFF', 'WIN%_DIFF', 'SEED_DIFF'
]
X_top = games_clean[top_features]
y     = games_clean['TARGET']

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

years = sorted(games_clean['YEAR'].unique())
gbm_brier = []
gbm_acc   = []
rf_brier  = []

for test_year in years:
    train_mask = games_clean['YEAR'] != test_year
    test_mask  = games_clean['YEAR'] == test_year

    X_train, X_test = X_top[train_mask], X_top[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    gbm = GradientBoostingClassifier(
        n_estimators=500, max_depth=4,
        learning_rate=0.05, subsample=0.8,
        random_state=42
    )
    gbm.fit(X_train, y_train)
    gbm_probs = gbm.predict_proba(X_test)[:, 1]
    gbm_preds = gbm.predict(X_test)
    gbm_brier.append(brier_score_loss(y_test, gbm_probs))
    gbm_acc.append((gbm_preds == y_test).mean())

    rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_brier.append(brier_score_loss(y_test, rf.predict_proba(X_test)[:, 1]))

    print(f"{test_year}  GBM Brier: {gbm_brier[-1]:.4f}  RF Brier: {rf_brier[-1]:.4f}  Acc: {gbm_acc[-1]:.3f}")

print(f"\nMean GBM Brier: {np.mean(gbm_brier):.4f}")
print(f"Mean RF Brier:  {np.mean(rf_brier):.4f}")

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
import matplotlib.pyplot as plt

years = sorted(games_clean['YEAR'].unique())
rf_cal_brier  = []
rf_raw_brier  = []

for test_year in years:
    train_mask = games_clean['YEAR'] != test_year
    test_mask  = games_clean['YEAR'] == test_year

    X_train, X_test = X_top[train_mask], X_top[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    # Raw RF
    rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    raw_probs = rf.predict_proba(X_test)[:, 1]
    rf_raw_brier.append(brier_score_loss(y_test, raw_probs))

    # Calibrated RF (Platt scaling via sigmoid)
    rf_cal = CalibratedClassifierCV(
        RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1),
        method='sigmoid', cv=5
    )
    rf_cal.fit(X_train, y_train)
    cal_probs = rf_cal.predict_proba(X_test)[:, 1]
    rf_cal_brier.append(brier_score_loss(y_test, cal_probs))

    print(f"{test_year}  Raw RF: {rf_raw_brier[-1]:.4f}  Calibrated RF: {rf_cal_brier[-1]:.4f}")

print(f"\nMean Raw RF Brier:        {np.mean(rf_raw_brier):.4f}")
print(f"Mean Calibrated RF Brier: {np.mean(rf_cal_brier):.4f}")

# ── Calibration curve on 2025 holdout ─────────────────────────────────────────
train_mask = games_clean['YEAR'] != 2025
test_mask  = games_clean['YEAR'] == 2025

rf_cal_final = CalibratedClassifierCV(
    RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1),
    method='sigmoid', cv=5
)
rf_cal_final.fit(X_top[train_mask], y[train_mask])
probs_2025 = rf_cal_final.predict_proba(X_top[test_mask])[:, 1]

frac_pos, mean_pred = calibration_curve(y[test_mask], probs_2025, n_bins=8)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(mean_pred, frac_pos, 's-', label='Calibrated RF')
ax.plot([0,1],[0,1],'k--', label='Perfect')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration curve — 2025 holdout')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Check what 2025 data looks like in our dataset
games_2025 = games_clean[games_clean['YEAR'] == 2025].copy()
print(f"2025 games in dataset: {len(games_2025)}")
print(games_2025[['TEAM_A','SEED_A','TEAM_B','SEED_B','CURRENT_ROUND','TARGET']].to_string())

In [ ]:
# ── 1. Train final model on 2008-2024 ─────────────────────────────────────────
train_mask = games_clean['YEAR'] != 2025
test_mask  = games_clean['YEAR'] == 2025

rf_final = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf_final.fit(X_top[train_mask], y[train_mask])

# ── 2. Predict 2025 ────────────────────────────────────────────────────────────
probs_2025 = rf_final.predict_proba(X_top[test_mask])[:, 1]
preds_2025 = rf_final.predict(X_top[test_mask])

results_2025 = games_2025[['CURRENT_ROUND','TEAM_A','SEED_A','TEAM_B','SEED_B','TARGET']].copy()
results_2025['PRED_PROB_A']  = probs_2025.round(3)
results_2025['PREDICTED_WIN'] = preds_2025
results_2025['CORRECT']       = (preds_2025 == results_2025['TARGET']).astype(int)
results_2025['ROUND_NAME'] = results_2025['CURRENT_ROUND'].map({
    64: 'Round of 64', 32: 'Round of 32',
    16: 'Sweet 16', 8: 'Elite 8', 4: 'Final Four', 2: 'Championship'
})

# ── 3. Summary by round ────────────────────────────────────────────────────────
print("=== Accuracy by Round ===")
round_order = ['Round of 64','Round of 32','Sweet 16','Elite 8','Final Four','Championship']
for rnd in round_order:
    subset = results_2025[results_2025['ROUND_NAME'] == rnd]
    if len(subset) == 0:
        continue
    acc = subset['CORRECT'].mean()
    print(f"{rnd:20s}  {subset['CORRECT'].sum()}/{len(subset)}  ({acc:.1%})")

brier_2025 = brier_score_loss(results_2025['TARGET'], probs_2025)
print(f"\n2025 Brier Score: {brier_2025:.4f}")
print(f"2025 Overall Accuracy: {results_2025['CORRECT'].mean():.1%}")

# ── 4. Print upsets we got wrong and right ────────────────────────────────────
print("\n=== Upsets (higher seed won) ===")
upsets = results_2025[results_2025['TARGET'] == 0].copy()  # team B won
upsets['MODEL_CAUGHT'] = (upsets['PREDICTED_WIN'] == 0).astype(int)
print(upsets[['ROUND_NAME','TEAM_A','SEED_A','TEAM_B','SEED_B',
              'PRED_PROB_A','MODEL_CAUGHT']].to_string())

# ── 5. Full results sorted by round ───────────────────────────────────────────
print("\n=== Full 2025 Predictions vs Actuals ===")
print(results_2025.sort_values('CURRENT_ROUND', ascending=False)[
    ['ROUND_NAME','TEAM_A','SEED_A','TEAM_B','SEED_B',
     'PRED_PROB_A','TARGET','CORRECT']
].to_string())

In [ ]:
# ── Final model summary + save everything ─────────────────────────────────────
import pickle

# Save the trained model and feature list
with open('rf_march_madness_final.pkl', 'wb') as f:
    pickle.dump({'model': rf_final, 'features': top_features}, f)

# ── Pretty summary table ───────────────────────────────────────────────────────
print("=" * 55)
print("  MARCH MADNESS MODEL — FINAL RESULTS SUMMARY")
print("=" * 55)
print(f"\n  Model:         Random Forest (500 trees)")
print(f"  Training data: 2008–2024 ({train_mask.sum()} games)")
print(f"  Features:      {len(top_features)} efficiency diff features")
print(f"\n  ── Cross-validated performance (2008–2025) ──")
print(f"  Mean Brier Score:  0.1917  (target < 0.20) ✓")
print(f"  Mean Accuracy:     69.4%")
print(f"\n  ── 2025 holdout performance ──")
print(f"  Brier Score:  {brier_2025:.4f}  ✓")
print(f"  Accuracy:     {results_2025['CORRECT'].mean():.1%}")

print(f"\n  ── 2025 accuracy by round ──")
for rnd in round_order:
    subset = results_2025[results_2025['ROUND_NAME'] == rnd]
    if len(subset) == 0: continue
    acc = subset['CORRECT'].mean()
    bar = '█' * int(acc * 20)
    print(f"  {rnd:20s} {subset['CORRECT'].sum()}/{len(subset)}  {bar} {acc:.0%}")

print(f"\n  Top predictors:")
for feat, imp in importances.head(5).items():
    print(f"    {feat:20s}  {imp:.4f}")
print("=" * 55)

# ── Worst misses (games we were most wrong about) ─────────────────────────────
results_2025['ERROR'] = abs(results_2025['PRED_PROB_A'] - results_2025['TARGET'])
print("\nTop 5 worst predictions:")
print(results_2025.nlargest(5, 'ERROR')[
    ['ROUND_NAME','TEAM_A','SEED_A','TEAM_B','SEED_B','PRED_PROB_A','TARGET']
].to_string())

In [ ]:
import xgboost as xgb
print(xgb.__version__)

In [ ]:
import xgboost as xgb
from sklearn.metrics import brier_score_loss
import numpy as np

# ── Need to re-define these after kernel restart ───────────────────────────────
# Re-run your data pipeline cells first (the ones that build games_clean, X_top, y)
# Then run this cell

years = sorted(games_clean['YEAR'].unique())
xgb_brier = []
rf_brier  = []
xgb_acc   = []

for test_year in years:
    train_mask = games_clean['YEAR'] != test_year
    test_mask  = games_clean['YEAR'] == test_year

    X_train, X_test = X_top[train_mask], X_top[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    # XGBoost
    xgb_model = xgb.XGBClassifier(
        n_estimators=500, max_depth=4,
        learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, eval_metric='logloss',
        random_state=42, n_jobs=-1
    )
    xgb_model.fit(X_train, y_train)
    xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
    xgb_brier.append(brier_score_loss(y_test, xgb_probs))
    xgb_acc.append((xgb_model.predict(X_test) == y_test).mean())

    # RF for comparison
    rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_probs = rf.predict_proba(X_test)[:, 1]
    rf_brier.append(brier_score_loss(y_test, rf_probs))

    print(f"{test_year}  XGB: {xgb_brier[-1]:.4f}  RF: {rf_brier[-1]:.4f}  XGB Acc: {xgb_acc[-1]:.3f}")

print(f"\nMean XGB Brier: {np.mean(xgb_brier):.4f}")
print(f"Mean RF Brier:  {np.mean(rf_brier):.4f}")

# ── Ensemble (blend RF + XGB) ──────────────────────────────────────────────────
ens_brier = []
for test_year in years:
    train_mask = games_clean['YEAR'] != test_year
    test_mask  = games_clean['YEAR'] == test_year

    X_train, X_test = X_top[train_mask], X_top[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    xgb_m = xgb.XGBClassifier(
        n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42, n_jobs=-1
    )
    xgb_m.fit(X_train, y_train)

    rf_m = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
    rf_m.fit(X_train, y_train)

    ensemble_probs = 0.5 * xgb_m.predict_proba(X_test)[:, 1] + \
                     0.5 * rf_m.predict_proba(X_test)[:, 1]
    ens_brier.append(brier_score_loss(y_test, ensemble_probs))

print(f"Mean Ensemble Brier: {np.mean(ens_brier):.4f}")
print(f"\nBest model: ", end="")
scores = {'RF': np.mean(rf_brier), 'XGB': np.mean(xgb_brier), 'Ensemble': np.mean(ens_brier)}
print(min(scores, key=scores.get), f"({min(scores.values()):.4f})")

In [ ]:
import pickle

with open('rf_march_madness_2026.pkl', 'wb') as f:
    pickle.dump({
        'model'   : rf_final,
        'features': top_features
    }, f)

games_clean.to_csv('games_clean_2026.csv', index=False)

print("Model saved — ready for 2026 bracket!")
print(f"Features used: {top_features}")
print(f"Expected Brier score: ~0.19")